# BDACA II — Wikipedia Linking

## Advanced MediaWiki API, Caching, and Wikidata as a Join Key

### What you will learn

| #   | Topic                           | Key skill                                         |
| --- | ------------------------------- | ------------------------------------------------- |
| 1   | Advanced MediaWiki API          | Batching, combined props, pagination, rate limits |
| 2   | Caching & constrained computing | Reproducibility under API quotas                  |
| 3   | Wikidata as a join key          | Linking Wikipedia to external datasets            |


### Shared imports and module-level session

All code uses only packages in the course environment. We define a
`requests.Session` here at module level — one object that persists for the
lifetime of the kernel, maintains a connection pool, and automatically attaches
our `User-Agent` header to every request.


In [1]:
import requests
import json
import time
import itertools
import random
from pathlib import Path
import pandas as pd

# ── Module-level session ──────────────────────────────────────────────────────
# A Session reuses the underlying TCP connection across calls (connection pooling)
# and lets us set headers once rather than repeating them in every requests.get().
SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "BDACA-II-course/1.0 (educational; contact@example.com)"
})

API = "https://en.wikipedia.org/w/api.php"   # MediaWiki JSON API base URL

print(f"requests {requests.__version__}")
print(f"pandas   {pd.__version__}")
print(f"Session ready, User-Agent: {SESSION.headers['User-Agent']}")

requests 2.32.5
pandas   2.3.3
Session ready, User-Agent: BDACA-II-course/1.0 (educational; contact@example.com)


## Topic 1 — Advanced MediaWiki API Usage

The MediaWiki API (`en.wikipedia.org/w/api.php`) is the primary entry point for
programmatic Wikipedia research. Four mistakes account for most scraping bugs
and rate-limit bans:

1. **One API call per title** — the API accepts up to 50 at once.
2. **Separate calls per data type** — `prop` accepts multiple modules in one
   call.
3. **Ignoring continuation tokens** — multi-page results are silently truncated.
4. **Missing User-Agent and maxlag** — Wikimedia will throttle or block the
   client.

We address each in turn, then show the two additional endpoints students most
often need: redirect/normalization resolution and the Pageviews REST API.


### 1.1 The naive loop — what not to do

The pattern below is the instinctive first attempt. It works, but it issues _N_
HTTP requests for _N_ articles. For a study of 500 Dutch politicians that is 500
round-trips when 10 would suffice.


In [2]:
# ── WRONG WAY ──────────────────────────────────────────────────────────────────
# One API call per title — unacceptable at scale.

titles_sample = ["Mark Rutte", "Geert Wilders", "Jesse Klaver"]

for t in titles_sample:          # 3 titles → 3 HTTP calls
    r = SESSION.get(
        API,
        params={"action": "query", "titles": t, "prop": "info", "format": "json"},
        timeout=10,
    )
    r.raise_for_status()
    # Note: without formatversion=2, pages is a dict keyed by pageid, not a list
    # — this is another reason the naive approach is harder to work with.
    # Every other cell in this notebook uses formatversion=2 to get a clean list.
    for pid, page in r.json()["query"]["pages"].items():
        print(f"{page.get('title', '?'):40s}  pageid={pid}")


Mark Rutte                                pageid=4321056
Geert Wilders                             pageid=1191676
Jesse Klaver                              pageid=32171498


### 1.2 The production wrapper: `api_get()`

Every MediaWiki API call in this notebook goes through one wrapper function. Run
the cell below, then read what it provides and why.


In [3]:
def api_get(params: dict, api: str = API, max_retries: int = 5, maxlag: int = 5) -> dict:
    """GET the MediaWiki JSON API via the module-level session.

    Features
    --------
    - Adds `maxlag` to every request (cooperative rate limiting).
    - Exponential back-off on 429/503/504, capped at 60 s.
    - Raises RuntimeError on API-level errors (HTTP 200 with error body).
    - Raises RuntimeError after max_retries exhausted.
    """
    # Create an internal copy so we never mutate the caller's dict.
    call_params = {**params, "maxlag": str(maxlag)}

    wait = 2  # seconds; doubles each retry, capped at 60
    for attempt in range(1, max_retries + 1):
        try:
            r = SESSION.get(api, params=call_params, timeout=15)

            if r.status_code in (429, 503, 504):
                print(f"  HTTP {r.status_code} (attempt {attempt}), waiting {wait}s ...")
                time.sleep(wait)
                wait = min(wait * 2, 60)   # cap at 60 s
                continue

            r.raise_for_status()
            result = r.json()

            if "error" in result:
                err = result["error"]
                raise RuntimeError(
                    f"MediaWiki API error [{err.get('code', '?')}]: {err.get('info', err)}"
                )
            if "errors" in result:
                msgs = "; ".join(e.get("text", str(e)) for e in result["errors"])
                raise RuntimeError(f"MediaWiki API errors: {msgs}")

            return result

        except requests.exceptions.Timeout as exc:
            print(f"  Timeout on attempt {attempt}, waiting {wait}s ...")
            time.sleep(wait)
            wait = min(wait * 2, 60)

        except requests.exceptions.ConnectionError as exc:
            print(f"  Connection error (attempt {attempt}): {exc!r}, waiting {wait}s ...")
            time.sleep(wait)
            wait = min(wait * 2, 60)

    raise RuntimeError(f"API call failed after {max_retries} retries")

data = api_get({"action": "query", "titles": "Tweede Kamer", "prop": "info",
                "format": "json", "formatversion": "2"})
p = data["query"]["pages"][0]
print(f"Title:  {p['title']}")
print(f"Length: {p['length']:,} chars")
print(f"Touched:{p['touched']}")

Title:  Tweede Kamer
Length: 52 chars
Touched:2026-03-02T11:41:23Z


The wrapper provides three things that every production API client needs:

| Feature                                   | Why                                                                                                                                                                                            |
| ----------------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **`requests.Session`**                    | Reuses TCP connections; User-Agent set once for all calls.                                                                                                                                     |
| **Exponential back-off (capped at 60 s)** | Handles transient `429`, `503`, `504` without crashing.                                                                                                                                        |
| **`maxlag` parameter**                    | Cooperative throttle required by [Wikimedia bot policy](https://www.mediawiki.org/wiki/Manual:Maxlag_parameter). Returns `503` when the db replica is behind; triggers back-off automatically. |

The User-Agent header is the most overlooked requirement. Wikimedia's
[policy](https://meta.wikimedia.org/wiki/User-Agent_policy) requires it to
identify your tool and include contact info. Requests without it are throttled;
repeated violations result in IP blocks.


### 1.3 Batching up to 50 titles per call

The `titles` parameter accepts a **pipe-separated list of up to 50 titles**.
`formatversion=2` returns `pages` as a list rather than a dict keyed by
`pageid`, which is cleaner to iterate. Pages that do not exist get a `missing`
key in their object, so you can detect bad titles without extra calls.


In [4]:
POLITICIANS = [
    "Mark Rutte", "Geert Wilders", "Jesse Klaver", "Sigrid Kaag",
    "Wopke Hoekstra", "Lilian Marijnissen", "Thierry Baudet",
    "VVD", "PVV", "D66", "GroenLinks", "SP",
    "Labour Party (Netherlands)", "ChristenUnie",
]

data = api_get({
    "action":        "query",
    "titles":        "|".join(POLITICIANS),   # pipe-separated, max 50
    "prop":          "info",
    "inprop":        "url",
    "format":        "json",
    "formatversion": "2",                     # pages is a list, not a dict
})

for page in data["query"]["pages"]:
    if page.get("missing"):
        status = "MISSING"
    else:
        status = f"id={page['pageid']}, {page.get('length', 0):,} chars"
    print(f"{page.get('title', '?'):45s}  {status}")

Labour Party (Netherlands)                     id=49066, 56,938 chars
GroenLinks                                     id=155948, 79,511 chars
D66                                            id=172354, 596 chars
SP                                             id=330177, 5,866 chars
PVV                                            id=437740, 808 chars
Geert Wilders                                  id=1191676, 210,687 chars
Mark Rutte                                     id=4321056, 102,888 chars
ChristenUnie                                   id=6667531, 132 chars
Jesse Klaver                                   id=32171498, 20,100 chars
Thierry Baudet                                 id=40321354, 62,360 chars
Sigrid Kaag                                    id=40791438, 52,177 chars
Wopke Hoekstra                                 id=55783045, 38,131 chars
Lilian Marijnissen                             id=56106493, 10,511 chars
VVD                                            id=71761249, 123 chars


### 1.4 Combined `prop` modules — one round-trip, multiple data types

The `prop` parameter accepts a **pipe-separated list of modules**. This is one
of the most impactful optimisations available: instead of three separate API
calls for revision history, categories, and page properties, you issue one and
parse the combined response.


In [5]:
data = api_get({
    "action":        "query",
    "titles":        "Mark Rutte|Geert Wilders|Jesse Klaver",
    "prop":          "revisions|categories|pageprops",   # three modules, one call
    "rvprop":        "ids|timestamp|user|size",          # which revision fields
    # NOTE: rvlimit is not allowed when titles contains multiple pages;
    # we take only [0] below, so omitting it has no effect on the output.
    "cllimit":       "50",                               # up to 50 categories (anon limit)
    "ppprop":        "wikibase_item",                    # Wikidata QID
    "format":        "json",
    "formatversion": "2",
})

for page in data["query"]["pages"]:
    rev  = (page.get("revisions") or [{}])[0]
    cats = [c["title"].replace("Category:", "") for c in page.get("categories", [])][:4]
    qid  = page.get("pageprops", {}).get("wikibase_item", "—")

    # rev["user"] and rev["size"] can be absent if the edit was oversight-suppressed
    editor = rev.get("user") or "(suppressed)"
    size   = rev.get("size", 0)

    print(f"\n{page['title']}")
    print(f"  QID      : {qid}")
    print(f"  Last edit: {rev.get('timestamp', '?')}  by {editor}  ({size:,} bytes)")
    print(f"  Categories (first 4): {cats}")


Geert Wilders
  QID      : Q105338
  Last edit: 2026-02-26T04:21:52Z  by Gamerwierdo100  (210,687 bytes)
  Categories (first 4): ['1963 births', '20th-century Dutch politicians', '21st-century Dutch politicians', 'All articles lacking reliable references']

Mark Rutte
  QID      : Q57792
  Last edit: 2026-02-26T08:51:40Z  by Shacharrz  (102,888 bytes)
  Categories (first 4): []

Jesse Klaver
  QID      : Q2354753
  Last edit: 2026-02-03T19:18:59Z  by InternetArchiveBot  (20,100 bytes)
  Categories (first 4): []


### 1.5 Handling continuation tokens

Many API modules return results in pages (e.g. `pllimit` defaults to 10 for
`pagelinks`, 50 for `categories`). If you ignore the `continue` key in the
response you **silently get only the first page** — a common source of
undercount bugs in link-network and category-coverage studies.

**Pattern:** loop until `"continue"` is absent, merging continuation parameters
into the next call with `params.update(data["continue"])`. The `continue` block
contains the exact parameters needed to resume — do not compute them manually.


In [6]:
def get_all_pagelinks(title: str) -> list:
    """Retrieve ALL internal links from a Wikipedia article, following continuation tokens."""
    params = {
        "action":        "query",
        "titles":        title,
        "prop":          "links",
        "pllimit":       "max",    # 50 for anon users
        "plnamespace":   "0",      # article namespace only
        "format":        "json",
        "formatversion": "2",
    }
    all_links  = []
    page_count = 0

    while True:
        data = api_get(params)   # api_get adds maxlag internally each call
        page_count += 1

        for page in data["query"]["pages"]:
            all_links.extend(lk["title"] for lk in page.get("links", []))

        # ── The critical check ────────────────────────────────────────────────
        if "continue" not in data:
            break                               # no more pages
        params.update(data["continue"])         # merge continuation params and loop
        # ─────────────────────────────────────────────────────────────────────

    print(f"'{title}': {len(all_links)} links in {page_count} API call(s)")
    return all_links

links = get_all_pagelinks("2023 Dutch general election")
print(f"First 10: {links[:10]}")

'2023 Dutch general election': 428 links in 1 API call(s)
First 10: ['1797 Batavian Republic constitutional referendum', '1798 Batavian Republic constitutional referendum', '1801 Batavian Republic constitutional referendum', '1805 Batavian Republic constitutional referendum', '1848 Dutch general election', '1850 Dutch general election', '1852 Dutch general election', '1853 Dutch general election', '1854 Dutch general election', '1856 Dutch general election']


### 1.6 Redirect and normalization — the silent merge-killer

When your code sends a title like `"mark rutte"` or `"D66"`, the API may
silently transform it before looking up the page:

- **Normalization:** fixes capitalisation and replaces underscores with spaces.
  `"mark rutte"` → `"Mark Rutte"`. The `normalized` block in the response
  records every transformation.
- **Redirect resolution:** when the requested title is a redirect, the API can
  follow it automatically (`redirects=1`) and report the chain in the
  `redirects` block.

**Why this matters for merging:** if you built your dataset with one title form
and your API response used another, a plain `merge()` on `en_title` will
silently drop rows. Always resolve titles to their canonical form before
joining.


In [7]:
def resolve_canonical(titles: list) -> dict:
    """Return a mapping {input_title: canonical_title} after normalization + redirects.

    Uses redirects=1 so the API follows redirect chains automatically.
    Any title that could not be resolved maps to itself.
    """
    data = api_get({
        "action":        "query",
        "titles":        "|".join(titles),
        "redirects":     1,
        "format":        "json",
        "formatversion": "2",
    })
    query = data.get("query", {})

    # Build single-step lookup tables from the API's own chain data.
    norm_map  = {n["from"]: n["to"] for n in query.get("normalized", [])}
    redir_map = {r["from"]: r["to"] for r in query.get("redirects",  [])}

    mapping = {}
    for title in titles:
        step1 = norm_map.get(title, title)       # normalization (capitalisation, underscores)
        step2 = redir_map.get(step1, step1)      # redirect resolution
        mapping[title] = step2

    return mapping


# Demo: lowercase, underscore, and redirect cases
test_inputs = [
    "mark rutte",        # lowercase → normalized
    "Jesse_Klaver",      # underscore → normalized
    "D66",               # redirect to "Democrats 66"
    "VVD",               # redirect to "People's Party for Freedom and Democracy"
    "Tweede Kamer",      # canonical — unchanged
]

canonical = resolve_canonical(test_inputs)

print(f"{'Input':35s}  {'Result':45s}  {'Change'}")
print("-" * 95)
for inp, out in canonical.items():
    if inp == out:
        change = "unchanged"
    elif inp.lower().replace("_", " ") == out.lower():
        change = "normalized"        # only capitalisation or underscores changed
    else:
        change = "REDIRECTED"        # content changed — merge on original title would silently fail
    print(f"{inp:35s}  {out:45s}  {change}")


Input                                Result                                         Change
-----------------------------------------------------------------------------------------------
mark rutte                           Mark Rutte                                     normalized
Jesse_Klaver                         Jesse Klaver                                   normalized
D66                                  D66                                            unchanged
VVD                                  People's Party for Freedom and Democracy       REDIRECTED
Tweede Kamer                         House of Representatives (Netherlands)         REDIRECTED


## Extra: Disambiguation Pages


In [8]:
def find_disambig(titles: list) -> list:
    """Return any titles in the list that point to disambiguation pages."""
    data = api_get({
        "action":        "query",
        "titles":        "|".join(titles),
        "prop":          "pageprops",
        "ppprop":        "disambiguation",
        "format":        "json",
        "formatversion": "2",
    })
    return [
        page["title"]
        for page in data["query"]["pages"]
        if "disambiguation" in page.get("pageprops", {})
    ]

# Good practice: run this on your canonical titles before any collection
canonical = resolve_canonical(test_inputs)
disambig  = find_disambig(list(canonical.values()))

if disambig:
    print("Disambiguation pages found — fix these before collecting:")
    for t in disambig:
        print(f"   {t}")
else:
    print("No disambiguation pages.")

Disambiguation pages found — fix these before collecting:
   D66


Disambiguation page found?

- You have QIDs from SPARQL → use en_title from df_mps directly (§3.1)
- Small dataset (<50 titles) → MANUAL_CORRECTIONS dict (see below)
- Large dataset → get_disambig_targets() + filter by known keywords (e.g.
  "politician", "minister") (see below)


In [9]:
MANUAL_CORRECTIONS = {
    "Jan de Vries": "Jan de Vries (politician)",   # disambig → specific article
    "Peter de Groot": "Peter de Groot (PvdA)",
}

In [10]:
def get_disambig_targets(title: str) -> list:
    """Return the linked articles from a disambiguation page."""
    data = api_get({
        "action":        "query",
        "titles":        "D66",
        "prop":          "links",
        "pllimit":       "max",
        "plnamespace":   "0",
        "format":        "json",
        "formatversion": "2",
    })
    page = data["query"]["pages"][0]
    return [lk["title"] for lk in page.get("links", [])]

# Example: inspect what "D66" disambiguates to
for t in disambig:
    candidates = get_disambig_targets(t)
    print(f"\n{t} → {len(candidates)} candidates:")
    for c in candidates:
        print(f"   {c}")


D66 → 7 candidates:
   D66 (die)
   D66 road (Croatia)
   D66 strain of Chlamydomonas reinhardtii
   Delta Junction Airport
   Democrats 66
   HMS Emerald (D66)
   Queen's Gambit Declined


### 1.7 The Wikimedia Pageviews REST API

Pageview data lives on a **separate endpoint** from the MediaWiki API:

```
https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/
    {project}/{access}/{agent}/{article}/{granularity}/{start}/{end}
```

| Parameter     | Common values                              |
| ------------- | ------------------------------------------ |
| `project`     | `en.wikipedia`, `nl.wikipedia`             |
| `access`      | `all-access`, `desktop`, `mobile-web`      |
| `agent`       | `all-agents`, `user` (excludes bots)       |
| `granularity` | `daily`, `monthly`                         |
| `start`/`end` | `YYYYMMDD` (daily) or `YYYYMM01` (monthly) |

The same `SESSION` applies — the User-Agent header is equally required here.
Article titles use underscores instead of spaces in the URL path.


In [11]:
def get_pageviews(
    article:     str,
    project:     str = "en.wikipedia",
    granularity: str = "monthly",
    start:       str = "20230101",
    end:         str = "20231201",
    access:      str = "all-access",
    agent:       str = "user",          # "user" excludes bot traffic
) -> list:
    """Fetch pageview data from the Wikimedia Pageviews REST API.
    Returns a list of dicts with keys: project, article, granularity, timestamp, access, agent, views.
    """
    article_slug = article.replace(" ", "_")   # REST API uses underscores
    url = (
        f"https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article"
        f"/{project}/{access}/{agent}/{article_slug}/{granularity}/{start}/{end}"
    )
    r = SESSION.get(url, timeout=15)
    r.raise_for_status()
    return r.json().get("items", [])

# Fetch 2023 monthly views for three Dutch politicians
politicians = ["Mark Rutte", "Geert Wilders", "Jesse Klaver"]

rows = []
for politician in politicians:
    items = get_pageviews(politician, start="20230101", end="20231201")
    for item in items:
        rows.append({
            "name":      politician,
            "month":     pd.to_datetime(item["timestamp"], format="%Y%m%d%H").strftime("%Y-%m"),   # "2023/01/01 00:00:00" -> "2023/01"
            "views":     item["views"],
        })

df_views = pd.DataFrame(rows)

# Pivot to wide format for easy comparison
pivot = df_views.pivot(index="month", columns="name", values="views")
print(pivot.to_string())
print(f"\nAnnual totals:")
print(df_views.groupby("name")["views"].sum().sort_values(ascending=False).to_string())

name     Geert Wilders  Jesse Klaver  Mark Rutte
month                                           
2023-01          11846          1178       28341
2023-02          10454          1296       24074
2023-03          16618          3386       37192
2023-04          11453          1552       27631
2023-05          17279          1143       28914
2023-06          17592          1191       29094
2023-07          27983          6539      177254
2023-08          15315          3572       50256
2023-09          18361          2235       36741
2023-10          19625          2328       34260
2023-11         652914         11248      119777
2023-12          17094           175        3576

Annual totals:
name
Geert Wilders    836534
Mark Rutte       597110
Jesse Klaver      35843


> **Note on async requests:** This notebook uses sequential `requests` calls
> intentionally. Async collection with `asyncio` / `aiohttp` can be 10–20×
> faster but adds complexity that makes debugging harder when you are still
> learning the API. Get the sequential pipeline working and validated first.
> Async is covered in the lecture slides — come back to it once you have
> confirmed your data looks correct.


## Topic 2 — Caching and Constrained Computing

When collecting data for a research project you face a hard constraint: **you
can only run the full collection once or twice** before you exceed your
polite-use quota. That means:

- Every API response should be written to disk **immediately** after receipt.
- Re-running the notebook should read from disk, not re-hit the API.
- Sampling must be reproducible and stratified _before_ collection begins.

Two caching patterns appear in practice:

- **One `.json` file per entity** — good for random access and small
  collections.
- **JSONL (newline-delimited JSON)** — better for large sequential collections
  because it is append-only: a crash preserves everything written so far.


### 2.1 File-based caching — one `.json` per entity

**Pattern:** before each API call, check whether `cache/{key}.json` exists. If
it does, load from disk. If not, fetch _and immediately save_.

Saving before any downstream processing means a crash mid-collection does not
lose already-fetched data — on the next run, those files are already there.


In [12]:
CACHE_DIR = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)

def cache_key(title: str) -> str:
    """Convert a Wikipedia title to a safe filename stem."""
    return title.replace("/", "_").replace(" ", "_").replace(":", "_")

def load_or_fetch(key: str, fetch_fn, **kwargs) -> dict:
    """Return cached result if present; otherwise call fetch_fn and cache it.

    Parameters
    ----------
    key      : filename stem, e.g. a QID or sanitised page title
    fetch_fn : callable returning a JSON-serialisable dict
    **kwargs : forwarded to fetch_fn
    """
    cache_file = CACHE_DIR / f"{key}.json"

    if cache_file.exists():
        with cache_file.open() as f:
            return json.load(f)             # cache HIT — no API call

    result = fetch_fn(**kwargs)             # cache MISS — call the API
    with cache_file.open("w") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    return result

def fetch_page_info(title: str) -> dict:
    """Single-title page info, intended for use with load_or_fetch."""
    data = api_get({
        "action": "query", "titles": title,
        "prop": "info", "format": "json", "formatversion": "2",
    })
    return data["query"]["pages"][0]

# Run this cell twice — the second run should print CACHE for all titles.
for title in ["Mark Rutte", "Geert Wilders", "Jesse Klaver"]:
    key    = cache_key(title)
    cached = (CACHE_DIR / f"{key}.json").exists()   # check BEFORE load_or_fetch writes the file
    info   = load_or_fetch(key, fetch_page_info, title=title)
    source = "CACHE" if cached else "FETCHED"
    print(f"{title:30s}  {info.get('length', 0):>7,} chars  [{source}]")

print(f"\nCache files in {CACHE_DIR}/:")
for f in sorted(CACHE_DIR.glob("*.json")):
    print(f"  {f.name}")

Mark Rutte                      102,888 chars  [FETCHED]
Geert Wilders                   210,687 chars  [FETCHED]
Jesse Klaver                     20,100 chars  [FETCHED]

Cache files in cache/:
  Geert_Wilders.json
  Jesse_Klaver.json
  Mark_Rutte.json


### 2.2 The JSONL append pattern — crash-safe collection at scale

For large collection jobs (hundreds or thousands of articles), writing one
`.json` per entity creates thousands of tiny files and makes it hard to know
what you have already collected.

**JSONL (newline-delimited JSON):** one JSON object per line in a single file.
The file is opened in **append mode (`"a"`)** — each record is written
immediately, so a crash at item 347 of 500 loses nothing. On resume, read the
file to build a set of already-collected keys and skip them.

Compared to a single JSON array, JSONL never requires reading the whole file
back before writing; it is also easy to process with
`pandas.read_json(lines=True)`.


In [13]:
JSONL_PATH = Path("cache/collection.jsonl")

def append_jsonl(record: dict, path: Path = JSONL_PATH) -> None:
    """Append one JSON record as a single line. Safe to call after a crash and resume."""
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

def read_jsonl(path: Path = JSONL_PATH) -> list:
    """Read all records from a JSONL file. Returns [] if file does not exist."""
    if not path.exists():
        return []
    with path.open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def already_collected(path: Path = JSONL_PATH, key: str = "title") -> set:
    """Return the set of already-collected key values (e.g. titles or QIDs)."""
    return {rec[key] for rec in read_jsonl(path) if key in rec}

# ── Demo ──────────────────────────────────────────────────────────────────────
#     DEMO ONLY — Remove this line before using in your own collection:
#     it resets the file on every run and will delete previously collected data.
JSONL_PATH.unlink(missing_ok=True)   # fresh start for demo

titles = ["Mark Rutte", "Geert Wilders", "Jesse Klaver",
          "Sigrid Kaag", "Lilian Marijnissen"]
done   = already_collected()

for title in titles:
    if title in done:
        print(f"  SKIP (already collected): {title}")
        continue

    data = api_get({"action": "query", "titles": title, "prop": "info",
                    "format": "json", "formatversion": "2"})
    page = data["query"]["pages"][0]
    record = {
        "title":   page.get("title"),
        "pageid":  page.get("pageid"),
        "length":  page.get("length", 0),
        "touched": page.get("touched", ""),
    }
    append_jsonl(record)
    print(f"  SAVED: {record['title']:30s}  {record['length']:,} chars")

# Show the raw file — each line is one complete JSON object
print(f"\nRaw content of {JSONL_PATH}:")
print(JSONL_PATH.read_text(encoding="utf-8"))

print(f"Records in {JSONL_PATH}: {len(read_jsonl())}")
print("Re-run this cell — SKIP messages confirm crash-safety.")

# Load back with pandas — works directly from JSONL
df_collected = pd.read_json(JSONL_PATH, lines=True)
print(f"\nDataFrame from JSONL:\n{df_collected[['title','length']].to_string(index=False)}")


  SAVED: Mark Rutte                      102,888 chars
  SAVED: Geert Wilders                   210,687 chars
  SAVED: Jesse Klaver                    20,100 chars
  SAVED: Sigrid Kaag                     52,177 chars
  SAVED: Lilian Marijnissen              10,511 chars

Raw content of cache/collection.jsonl:
{"title": "Mark Rutte", "pageid": 4321056, "length": 102888, "touched": "2026-03-02T05:54:08Z"}
{"title": "Geert Wilders", "pageid": 1191676, "length": 210687, "touched": "2026-03-02T08:27:19Z"}
{"title": "Jesse Klaver", "pageid": 32171498, "length": 20100, "touched": "2026-03-02T09:02:38Z"}
{"title": "Sigrid Kaag", "pageid": 40791438, "length": 52177, "touched": "2026-03-02T06:01:40Z"}
{"title": "Lilian Marijnissen", "pageid": 56106493, "length": 10511, "touched": "2026-03-02T06:05:52Z"}

Records in cache/collection.jsonl: 5
Re-run this cell — SKIP messages confirm crash-safety.

DataFrame from JSONL:
             title  length
        Mark Rutte  102888
     Geert Wilders  2106

### 2.3 Stratified sampling before collection

You rarely want _all_ articles in a set — you want a **balanced sample across
categories** before you hit the API. Stratifying first means you pay the API
cost only for articles you actually need, and your sample is fixed and
reproducible from the start.


In [14]:
def stratified_sample(category_map: dict, n_per_category: int, seed: int = 42) -> dict:
    """Sample up to n_per_category titles from each category.

    Parameters
    ----------
    category_map     : {category_label: [title, ...]}
    n_per_category   : maximum sample per category
    seed             : random seed for reproducibility

    Returns a new dict with the same keys; values truncated to n_per_category.
    """
    rng    = random.Random(seed)
    result = {}
    for cat, titles in category_map.items():
        pool = list(titles)     # copy — do not mutate the input
        rng.shuffle(pool)
        result[cat] = pool[:n_per_category]
    return result

# All names below are Wikipedia article titles (English).
# "Dilan Yeşilgöz" uses the short form without the -Zegerius suffix —
# this is the actual Wikipedia page title and is intentionally consistent
# with the df_pageviews entries in §3.3 (which carry the same comment there).
ALL_POLITICIANS = {
    "VVD":        ["Mark Rutte", "Dilan Yeşilgöz", "Klaas Dijkhoff",
                   "Edith Schippers", "Halbe Zijlstra", "Stef Blok"],
    "PVV":        ["Geert Wilders", "Fleur Agema", "Martin Bosma",
                   "Dion Graus", "Barry Madlener"],
    "D66":        ["Sigrid Kaag", "Alexander Pechtold", "Rob Jetten",
                   "Kajsa Ollongren", "Pia Dijkstra"],
    "GroenLinks": ["Jesse Klaver", "Bram van Ojik", "Kathalijne Buitenweg",
                   "Suzanne Kröger"],
    "SP":         ["Lilian Marijnissen", "Jan Marijnissen", "Emile Roemer",
                   "Ronald van Raak"],
}

sample = stratified_sample(ALL_POLITICIANS, n_per_category=3)
for party, titles in sample.items():
    print(f"{party:12s} -> {titles}")


VVD          -> ['Edith Schippers', 'Dilan Yeşilgöz', 'Klaas Dijkhoff']
PVV          -> ['Dion Graus', 'Martin Bosma', 'Geert Wilders']
D66          -> ['Kajsa Ollongren', 'Alexander Pechtold', 'Rob Jetten']
GroenLinks   -> ['Bram van Ojik', 'Kathalijne Buitenweg', 'Suzanne Kröger']
SP           -> ['Ronald van Raak', 'Emile Roemer', 'Lilian Marijnissen']


### 2.4 The `chunks()` utility — splitting a list into API batches

Once you have your sampled titles you need to split them into groups of ≤ 50 for
the batch API calls from §1.3. This two-liner is one of the most-reused helpers
in Wikipedia data-collection code.

The cell below also shows the **full pipeline**: sample → chunk → batch fetch →
append to JSONL, with a skip-if-already-collected guard.


In [15]:
def chunks(lst: list, n: int):
    """Yield successive n-sized chunks from lst."""
    for i in range(0, len(lst), n):
        yield lst[i : i + n]

# ── Full pipeline ─────────────────────────────────────────────────────────────
all_titles = list(itertools.chain.from_iterable(sample.values()))
n_calls    = -(-len(all_titles) // 50)   # ceiling division

print(f"Titles to fetch : {len(all_titles)}")
print(f"API calls needed: {n_calls}  (batch size 50)")

PIPELINE_JSONL = Path("cache/pipeline.jsonl")
#     DEMO ONLY — Remove this line before using in your own collection:
#     it resets the file on every run and will delete previously collected data.
PIPELINE_JSONL.unlink(missing_ok=True)   # fresh for demo
done = already_collected(PIPELINE_JSONL)

for batch_num, batch in enumerate(chunks(all_titles, 50), start=1):
    # Skip titles already in JSONL
    to_fetch = [t for t in batch if t not in done]
    if not to_fetch:
        print(f"Batch {batch_num}: all {len(batch)} already collected, skipping")
        continue

    print(f"\nBatch {batch_num}: fetching {len(to_fetch)} titles")
    data = api_get({
        "action":        "query",
        "titles":        "|".join(to_fetch),
        "prop":          "info",
        "format":        "json",
        "formatversion": "2",
    })
    for page in data["query"]["pages"]:
        record = {
            "title":   page.get("title"),
            "pageid":  page.get("pageid"),
            "missing": page.get("missing", False),
            "length":  page.get("length", 0),
        }
        append_jsonl(record, PIPELINE_JSONL)
        status = "MISSING" if record["missing"] else f"{record['length']:,} chars"
        print(f"  {record['title']:40s}  {status}")

print(f"\nTotal records: {len(read_jsonl(PIPELINE_JSONL))}")


Titles to fetch : 15
API calls needed: 1  (batch size 50)

Batch 1: fetching 15 titles
  Geert Wilders                             210,687 chars
  Kathalijne Buitenweg                      11,481 chars
  Alexander Pechtold                        23,178 chars
  Dion Graus                                13,707 chars
  Martin Bosma                              21,128 chars
  Bram van Ojik                             10,889 chars
  Emile Roemer                              7,431 chars
  Edith Schippers                           17,074 chars
  Klaas Dijkhoff                            13,131 chars
  Ronald van Raak                           5,178 chars
  Kajsa Ollongren                           22,898 chars
  Lilian Marijnissen                        10,511 chars
  Rob Jetten                                37,231 chars
  Dilan Yeşilgöz                            39,760 chars
  Suzanne Kröger                            8,353 chars

Total records: 15


## Topic 3 — Wikidata as a Join Key + Linking to External Datasets

Wikipedia page titles are **unstable identifiers**: they change on rename,
differ across language editions, and redirect chains complicate matching.

**Wikidata QIDs** (e.g. `Q57792` for Mark Rutte) are stable,
language-independent entity identifiers — the correct join key for linking.

The pipeline in this topic:

```
Wikipedia title ──> QID  (via SPARQL bulk lookup or MediaWiki API fallback)
                    |
       External dataset keyed by QID
                    |
      pandas merge + missing-match diagnosis
```


### 3.1 Querying Wikidata with SPARQL

The [Wikidata Query Service](https://query.wikidata.org/) exposes a SPARQL
endpoint that returns structured data about entities in bulk — far more
efficient than resolving QIDs one at a time via the MediaWiki API.

The query retrieves Dutch members of parliament with their QID, name, gender,
party, and Wikipedia page titles in both English and Dutch.

**Required headers:** The endpoint needs
`Accept: application/sparql-results+json` and a descriptive User-Agent — same
policy as the MediaWiki API.


In [16]:
WIKIDATA_SPARQL = "https://query.wikidata.org/sparql"

SPARQL_DUTCH_MPS = """
SELECT DISTINCT
  ?person ?personLabel  ?genderLabel  ?partyLabel
  ?en_title  ?nl_title
WHERE {
  ?person wdt:P39 wd:Q18887908 .            # position held: member of Tweede Kamer
  OPTIONAL { ?person wdt:P21 ?gender . }  # sex/gender
  OPTIONAL { ?person wdt:P102 ?party .  } # party affiliation

  OPTIONAL {                              # English Wikipedia sitelink
    ?en_article schema:about ?person ;
                schema:isPartOf <https://en.wikipedia.org/> ;
                schema:name ?en_title .
  }
  OPTIONAL {                              # Dutch Wikipedia sitelink
    ?nl_article schema:about ?person ;
                schema:isPartOf <https://nl.wikipedia.org/> ;
                schema:name ?nl_title .
  }
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en,nl" . }
}
ORDER BY ?personLabel
LIMIT 1000
"""

def run_sparql(query: str, endpoint: str = WIKIDATA_SPARQL) -> list:
    """Execute a SPARQL SELECT query; return results as a list of flat dicts."""
    r = SESSION.get(    # SESSION already has User-Agent set
        endpoint,
        params={"query": query, "format": "json"},
        headers={"Accept": "application/sparql-results+json"},
        timeout=30,     # SPARQL queries involve server-side query planning and can
                        # take several seconds — higher than api_get's 15 s timeout.
    )
    r.raise_for_status()
    bindings = r.json()["results"]["bindings"]
    rows = []
    for b in bindings:
        rows.append({
            "qid":      b["person"]["value"].rsplit("/", 1)[-1],   # Q-number only
            "name":     b.get("personLabel", {}).get("value", ""),
            "gender":   b.get("genderLabel", {}).get("value", ""),
            "party":    b.get("partyLabel",  {}).get("value", ""),
            "en_title": b.get("en_title",    {}).get("value", ""),
            "nl_title": b.get("nl_title",    {}).get("value", ""),
        })
    return rows

df_mps = pd.DataFrame(run_sparql(SPARQL_DUTCH_MPS))
print(f"Retrieved {len(df_mps)} rows")
df_mps.head(50)


Retrieved 1000 rows


,qid,name,gender,party,en_title,nl_title
0,Q105338,Q105338,male,Party for Freedom,Geert Wilders,Geert Wilders
1,Q2599351,Q2599351,male,Anti-Revolutionary Party,,Barthold Jacob Lintelo de Geer van Jutphaas
2,Q2698848,Q2698848,male,Christian Historical Union,Jim Janssen van Raaij,Jim Janssen van Raaij
3,Q2698848,Q2698848,male,Christian Democratic Appeal,Jim Janssen van Raaij,Jim Janssen van Raaij
4,Q2698848,Q2698848,male,Pim Fortuyn List,Jim Janssen van Raaij,Jim Janssen van Raaij
5,Q712105,Q712105,male,Christian Democratic Appeal,Hans van den Broek,Hans van den Broek
6,Q712105,Q712105,male,Catholic People's Party,Hans van den Broek,Hans van den Broek
7,Q21547423,A. van Laer,male,,,
8,Q20017658,A. van Weel Azn.,male,,,
9,Q21547427,A.A. Reael,male,,,


### Extended version: current MPs only, with GROUP_CONCAT

The query above is what the slides show. It is good enough for most exercises.
The version below is needed when you want **only the current parliament** (not
every MP since 1815) and want to handle MPs who belong to multiple parties. Note
that this is very detailed SPARQL stuff that you definitely do not need to know
by heart -- but you do need to be able to see whether the data returned is what
you would like to have returned (e.g. expecting 150 MEPs and not 1000s)

| Addition                                | What it does                                                                                                                                   |
| --------------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------- |
| `p:P39 / ps:P39` instead of `wdt:P39`   | Gives access to the statement node so you can read qualifiers (start date, end date, parliamentary group)                                      |
| `wikibase:BestRank`                     | Among all P39 statements for a person, picks only the "winning" one — the active one if ranks are set, otherwise all normal-rank ones          |
| `MINUS { pq:P582 }` + birth-year filter | Drops anyone with a recorded end date; birth ≥ 1940 catches historical MPs whose end date was never entered                                    |
| `GROUP_CONCAT(?partyName)`              | Collapses multiple party rows per person into a single comma-separated string                                                                  |
| `rdfs:label` + `"mul"` language code    | Fetches labels directly — more reliable inside aggregates; `mul` catches items like Geert Wilders whose label is stored once for all languages |
| `pq:P4100`                              | Parliamentary group (fractie), stored as a qualifier on the P39 statement, not as a direct property of the person                              |


In [17]:
SPARQL_DUTCH_MPS_CURRENT = """
SELECT 
  ?person ?personLabel ?genderLabel 
  (GROUP_CONCAT(DISTINCT ?partyName; SEPARATOR=", ") AS ?parties)
  (GROUP_CONCAT(DISTINCT ?groupName; SEPARATOR=", ") AS ?parl_groups)
  ?en_title ?nl_title
WHERE {
  ?person p:P39 ?statement .
  ?statement a wikibase:BestRank ;
             ps:P39 wd:Q18887908 .
  MINUS { ?statement pq:P582 ?end_time . }

  # Birth year filters out 19th-century MPs; made OPTIONAL to catch MPs with no
  # birth date recorded (otherwise we lose ~1 current MP with missing P569).
  OPTIONAL { ?person wdt:P569 ?birth . }
  FILTER(!BOUND(?birth) || ?birth >= "1940-01-01"^^xsd:dateTime)

  OPTIONAL { ?person wdt:P21 ?gender . }

  OPTIONAL {
    ?person wdt:P102 ?party .
    ?party rdfs:label ?partyName .
    FILTER(LANG(?partyName) IN ("nl", "en", "mul"))
  }

  OPTIONAL {
    ?statement pq:P4100 ?group .
    ?group rdfs:label ?groupName .
    FILTER(LANG(?groupName) IN ("nl", "en", "mul"))
  }

  OPTIONAL {
    ?en_article schema:about ?person ;
                schema:isPartOf <https://en.wikipedia.org/> ;
                schema:name ?en_title .
  }
  OPTIONAL {
    ?nl_article schema:about ?person ;
                schema:isPartOf <https://nl.wikipedia.org/> ;
                schema:name ?nl_title .
  }
  # "mul" catches items like Wilders whose label is set as multilingual rather
  # than per-language — without it the service falls back to the bare QID.
  SERVICE wikibase:label { bd:serviceParam wikibase:language "nl,en,mul" . }
}
GROUP BY ?person ?personLabel ?genderLabel ?en_title ?nl_title
ORDER BY ?personLabel
"""

# ── This replaces the df_mps from the simple query in 3.1 above. ─────────────
# The extended query retrieves only current MPs and collapses multiple party
# memberships into a single row per person. WIKIDATA_SPARQL and run_sparql
# were already defined in 3.1 — no need to redefine them here.
df_mps = pd.DataFrame(run_sparql(SPARQL_DUTCH_MPS_CURRENT))
print(f"Retrieved {len(df_mps)} rows")
df_mps.head(50)


Retrieved 150 rows


,qid,name,gender,party,en_title,nl_title
0,Q105338,Geert Wilders,mannelijk,,Geert Wilders,Geert Wilders
1,Q17427994,Alexander Kops,mannelijk,,Alexander Kops,Alexander Kops
2,Q136554575,Alisha Müller,vrouwelijk,,Alisha Müller,
3,Q110552900,André Flach,mannelijk,,André Flach,André Flach
4,Q136554681,André Poortman,mannelijk,,,André Poortman
5,Q136554539,Ani Zalinyan,vrouwelijk,,Ani Zalinyan,Ani Zalinyan
6,Q42532175,Annabel Nanninga,vrouwelijk,,Annabel Nanninga,Annabel Nanninga
7,Q83364807,Anne-Marijke Podt,vrouwelijk,,Anne-Marijke Podt,Anne-Marijke Podt
8,Q136554453,Annelotte Lammers,vrouwelijk,,Annelotte Lammers,Annelotte Lammers
9,Q123176053,Annette Raijer,vrouwelijk,,Annette Raijer,Annette Raijer


### 3.2 Resolving a Wikipedia title to a Wikidata QID

When you start from a Wikipedia title (e.g. from a pageview log) and need the
QID to join against Wikidata, use
`action=query&prop=pageprops&ppprop=wikibase_item`. The `wikibase_item` page
property contains the QID directly.

Use this as a _fallback_ for titles missing from your SPARQL results (e.g.
non-politician articles, recently renamed pages).


In [18]:
def title_to_qid(title: str, lang: str = "en") -> str:
    """Resolve a Wikipedia article title to its Wikidata QID, or '' if not found."""
    api_url = f"https://{lang}.wikipedia.org/w/api.php"
    data = api_get(
        {
            "action":        "query",
            "titles":        title,
            "prop":          "pageprops",
            "ppprop":        "wikibase_item",
            "format":        "json",
            "formatversion": "2",
        },
        api=api_url,
    )
    for page in data.get("query", {}).get("pages", []):
        qid = page.get("pageprops", {}).get("wikibase_item", "")
        if qid:
            return qid
    return ""

test_titles = [
    "Mark Rutte", "Geert Wilders", "Sigrid Kaag",
    "Jesse Klaver", "European Parliament",
]
for t in test_titles:
    print(f"{t:40s}  ->  {title_to_qid(t)}")

Mark Rutte                                ->  Q57792
Geert Wilders                             ->  Q105338
Sigrid Kaag                               ->  Q15077856
Jesse Klaver                              ->  Q2354753
European Parliament                       ->  Q8889


### 3.3 Linking Wikipedia metrics to an external dataset via QID

A common research task: you have Wikipedia pageview data (keyed by article
title) and an external dataset (e.g. parliamentary voting records, keyed by
QID). To join them you need a title → QID mapping column.

We walk through:

1. Build a QID lookup from the SPARQL results (fast — no extra API calls).
2. Fall back to `title_to_qid()` for titles absent from SPARQL.
3. Left-merge: keep all pageview rows; voting data fills in where available.
4. Diagnose missing matches quantitatively.


In [19]:
# ── Synthetic data — replace with your real sources ──────────────────────────
# In practice df_pageviews comes from the Wikimedia Pageviews API (§1.7).
# Note: df_views from §1.7 has columns (name, month, views); to use it here
# you would first rename the columns to match this structure:
#   df_pageviews = df_views.rename(columns={"name": "en_title", "views": "monthly_views"})
# Then drop the "month" column or keep it as an additional join key if needed.

df_pageviews = pd.DataFrame({
    "en_title": [
        "Mark Rutte", "Geert Wilders", "Sigrid Kaag", "Jesse Klaver",
        "Lilian Marijnissen", "Wopke Hoekstra", "Thierry Baudet",
        "Dilan Yeşilgöz",                    # Wikipedia title has no -Zegerius suffix
        "Caroline van der Plas",
        "Marianne Thieme",                   # retired — may lack recent votes
        "Pim Fortuyn",                       # deceased — no current voting record
        "List of Dutch political parties",   # non-politician — no QID match expected
    ],
    "monthly_views": [45000, 82000, 31000, 27000, 19000, 22000, 35000,
                      28000, 41000, 15000, 67000, 8000],
})

# Synthetic voting records — keyed by Wikidata QID.
# Retired (Thieme), deceased (Fortuyn), and non-person pages are absent
# by design: they illustrate systematic rather than random missingness.
df_votes = pd.DataFrame({
    "qid":       ["Q57792",    "Q105338", "Q15077856", "Q2354753",
                  "Q1882072",  "Q14520235", "Q4462875", "Q28861002",
                  "Q106044088"],
    "yes_votes": [312, 87, 201, 178, 156, 289, 63, 275, 134],
    "no_votes":  [41,  256, 142, 165, 187, 54, 280, 68, 209],
})

df_pageviews.head(12)


,en_title,monthly_views
0,Mark Rutte,45000
1,Geert Wilders,82000
2,Sigrid Kaag,31000
3,Jesse Klaver,27000
4,Lilian Marijnissen,19000
5,Wopke Hoekstra,22000
6,Thierry Baudet,35000
7,Dilan Yeşilgöz,28000
8,Caroline van der Plas,41000
9,Marianne Thieme,15000


In [20]:
# ── Step 1: resolve titles to QIDs ───────────────────────────────────────────
# Use SPARQL results as a fast lookup; fall back to the API only for misses.
sparql_lookup = dict(zip(df_mps["en_title"], df_mps["qid"]))

def resolve_qid(title: str) -> str:
    """Try SPARQL cache first; fall back to MediaWiki API for misses.

    Note: resolve_canonical() from §1.6 is intentionally not called here.
    resolve_canonical() is for cleaning input titles *before* data collection
    (e.g. fixing capitalisation before fetching pageviews). resolve_qid() is
    for QID lookup *during* merging, where titles should already be canonical.
    Different pipeline stages, different tools.
    """
    return sparql_lookup.get(title) or title_to_qid(title)

df_pageviews["qid"] = df_pageviews["en_title"].apply(resolve_qid)

print("QID resolution:")
print(df_pageviews[["en_title", "qid"]].to_string(index=False))


QID resolution:
                       en_title        qid
                     Mark Rutte     Q57792
                  Geert Wilders    Q105338
                    Sigrid Kaag  Q15077856
                   Jesse Klaver   Q2354753
             Lilian Marijnissen   Q1882072
                 Wopke Hoekstra  Q14520235
                 Thierry Baudet   Q4462875
                 Dilan Yeşilgöz  Q28861002
          Caroline van der Plas Q106044088
                Marianne Thieme    Q463434
                    Pim Fortuyn    Q311697
List of Dutch political parties           


In [21]:
# ── Temporal alignment: daily → monthly resampling ───────────────────────────
# The Pageviews API can return daily granularity. External datasets (e.g.
# voting records, survey waves) are usually monthly or quarterly. Merge keys
# must match, so resample before joining.

# Simulate a daily pageview series for one politician (replace with real API data)
import numpy as np
rng = np.random.default_rng(42)
daily_index = pd.date_range("2023-01-01", "2023-12-31", freq="D")
df_daily = pd.DataFrame({
    "date":  daily_index,
    "views": rng.integers(800, 3000, size=len(daily_index)),
})

# Resample to monthly totals — one line
df_monthly = df_daily.resample("ME", on="date")["views"].sum().reset_index()
df_monthly["month"] = df_monthly["date"].dt.to_period("M").astype(str)

print(df_monthly[["month", "views"]].to_string(index=False))
print(f"\nDaily rows: {len(df_daily)}  →  Monthly rows: {len(df_monthly)}")


  month  views
2023-01  61815
2023-02  53192
2023-03  61050
2023-04  57180
2023-05  59381
2023-06  57384
2023-07  58695
2023-08  56823
2023-09  52875
2023-10  58719
2023-11  54701
2023-12  62908

Daily rows: 365  →  Monthly rows: 12


In [22]:
# ── Step 2: left merge — keep all pageview rows ───────────────────────────────
df_merged = df_pageviews.merge(df_votes, on="qid", how="left")

# ── Step 3: diagnose missing matches ──────────────────────────────────────────
n_total   = len(df_merged)
n_matched = df_merged["yes_votes"].notna().sum()
n_missing = n_total - n_matched

print(f"{'Merge summary':-<50}")
print(f"  Total pageview rows:          {n_total}")
print(f"  Matched to voting records:    {n_matched}")
print(f"  Unmatched (no voting record): {n_missing}")
print(f"  Match rate:                   {n_matched / n_total:.0%}")

print(f"\n{'Unmatched titles':-<50}")
unmatched = df_merged.loc[
    df_merged["yes_votes"].isna(),
    ["en_title", "qid", "monthly_views"]
]
print(unmatched.to_string(index=False))

print(f"\n{'Matched sample (first 5)':-<50}")
matched = df_merged[df_merged["yes_votes"].notna()].head(5)
print(matched[["en_title", "monthly_views", "yes_votes", "no_votes"]].to_string(index=False))

Merge summary-------------------------------------
  Total pageview rows:          12
  Matched to voting records:    9
  Unmatched (no voting record): 3
  Match rate:                   75%

Unmatched titles----------------------------------
                       en_title     qid  monthly_views
                Marianne Thieme Q463434          15000
                    Pim Fortuyn Q311697          67000
List of Dutch political parties                   8000

Matched sample (first 5)--------------------------
          en_title  monthly_views  yes_votes  no_votes
        Mark Rutte          45000      312.0      41.0
     Geert Wilders          82000       87.0     256.0
       Sigrid Kaag          31000      201.0     142.0
      Jesse Klaver          27000      178.0     165.0
Lilian Marijnissen          19000      156.0     187.0


### 3.4 Missing matches are systematic bias, not random noise

When a merge fails the temptation is to drop non-matching rows and continue.
**Resist this.** Missing matches in Wikipedia-to-external-dataset linkages are
almost never _missing at random_ (MAR). They are missing for structural reasons
that correlate with your variables of interest.

**Three concrete examples for political communication research:**

1. **Deceased and retired politicians** have Wikipedia articles — often with
   high pageviews due to historical interest — but no current voting records.
   Dropping them inflates the apparent media engagement of _active_ politicians
   relative to historical figures, biasing any analysis of how Wikipedia
   constructs political legacy or memorialisation.

2. **Fringe or newly elected MPs** often lack English Wikipedia articles, or
   have stubs below detection thresholds. Coverage skews toward established
   parties. Any analysis of party-level attention will _underestimate_ smaller
   parties — precisely where Wikipedia's editorial gatekeeping is most
   consequential for research on media representation and agenda-setting.

3. **Non-person articles** (party pages, election overviews, committee pages)
   appear in pageview logs and will never match a voting-record QID. If these
   pages attract disproportionate views around elections, analysing only
   person-matched rows systematically underestimates total political
   information-seeking and misrepresents what citizens actually read.

> **Practical implication:** Before dropping unmatched rows, _characterise_
> them. Are they older? Less viewed? Skewed by party or gender? Report this as a
> limitation and consider the direction of bias it introduces in your results.


---

## Scope Discipline Checklist

Answer all five questions **before** scaling up your data collection. If you
cannot answer any of them, pause and redesign.

| #   | Question                                                                                                                                                                        | Why it matters                                                                        |
| --- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------- |
| 1   | **How many API calls will my full collection require?** Calculate `ceil(N / 50)` for batch info calls; add continuation rounds per article for pagelinks.                       | Prevents accidental overload and lets you estimate wall-clock time before committing. |
| 2   | **Have I cached all intermediate results to disk?** Every API response should be saved to `cache/` (`.json` or JSONL) before any downstream processing.                         | A crash at step 4 of 6 should not require re-fetching steps 1–3.                      |
| 3   | **Is my sample stratified and fixed before collection starts?** Generate and save `sample_titles.json` _before_ the first API call.                                             | Ensures reproducibility; prevents scope creep and ad-hoc additions mid-collection.    |
| 4   | **Have I characterised the match rate and pattern of missing data?** Run the merge diagnostic from §3.3 before analysis, not after.                                             | Missing matches are systematic; undocumented bias can invalidate inference.           |
| 5   | **Does my User-Agent identify my project and include contact information?** The `SESSION` from the imports cell handles this automatically — verify it has not been overridden. | Required by Wikimedia policy; protects your IP from throttling and potential blocks.  |


## Final checkpoint — what you can do now

After this notebook, you should be able to:

- Make polite, robust Wikipedia data-collection requests: use a single session
  with a proper User-Agent, batch up to 50 titles per call, combine multiple
  prop modules in one request, and follow continuation tokens so you do not
  silently truncate results.

- Work reproducibly under API limits: cache responses immediately (file-based
  cache or JSONL append pattern), and re-run your notebook without re-hitting
  the API for already-collected items.

- Prepare titles for linking: resolve normalization and redirects to canonical
  titles, detect disambiguation pages before collecting, and understand why
  disambiguation is a high-risk source of wrong matches.

- Collect attention data correctly: retrieve Pageviews time series from the
  Pageviews REST API, choose granularity (monthly vs daily), and reshape results
  into a tidy table or a pivot suitable for plotting.

- Link across datasets using stable identifiers: extract/resolve Wikidata QIDs
  and use them as join keys for merging Wikipedia-derived data with external
  datasets; diagnose missing matches rather than dropping rows silently.


## Self-test (bring your own domain)

Pick a topic from your final project or a hobby domain (examples: Pokémon, music
artists, football clubs, films, scientific concepts). Create a list of 10–30
Wikipedia entities and run a mini-pipeline that produces two outputs and a short
reflection.

Choose your entities

- Make a Python list MY_TITLES with 10–30 Wikipedia titles

Canonicalise before collecting

- Resolve normalization + redirects so each input maps to a canonical title
  (keep both columns: input_title, canonical_title).

- Run a disambiguation check on the canonical titles.

- If you hit disambiguation pages, stop and resolve them (preferred: use
  QIDs/sitelinks from Wikidata; acceptable for small sets: manual corrections
  with a documented mapping).

Collect a small, cached dataset

- Batch requests where possible (do not loop “one request per title”).

- Cache immediately to disk (either one JSON per entity or a JSONL file).

- Build a tidy entity table df_entities with at least: input_title,
  canonical_title, missing, is_disambig, qid (if available), and one additional
  field you care about (e.g., length, last_edit_timestamp, category count).

Collect and compare pageviews

- Fetch pageviews for the canonical titles for a fixed time window (monthly is
  fine).

- Build a tidy time-series table df_views with title, month, views, then make
  one pivot table or plot comparing entities over time.

Reflect briefly (3–6 lines in a markdown cell)

- Which titles changed during canonicalisation, and why?

- Did you encounter disambiguation pages; how did you resolve them?

- Which entities were missing or had no QID, and what is your plan to handle
  them in a real project?
